# idd-travel-time · Walkthrough

A minimal end-to-end vignette showing how to compute and visualise a travel-time
isochrone from a single origin point.

**What this notebook covers**

1. Construction a `TravelTimeRequest`
2. Running `compute()` with the local osmnx backend (no API key required)
3. Inspecting the result GeoDataFrames
4. Rendering all four visualisation layers with `viz.render()`
5. Saving the map to HTML

> **Note:** All network queries hit the OpenStreetMap Overpass API. The first run
> for a given area downloads the local street graph — expect 10–30 s depending on
> network speed and isochrone size. Subsequent runs over the same area are faster
> because osmnx caches the graph in memory within a session.


## 1 · Imports

In [ ]:
from idd_travel_time import viz
from idd_travel_time.constants import TimeContext, TransportMode, VizLayer
from idd_travel_time.isochrone import compute
from idd_travel_time.models import TravelTimeRequest

## 2 · Define the request

We want to know every place reachable **on foot within 20 minutes** from the
main entrance of Kenyatta National Hospital in Nairobi.


In [ ]:
request = TravelTimeRequest(
    origin_lat=-1.3006,
    origin_lon=36.8073,
    mode=TransportMode.WALKING,
    duration_minutes=20,
    time_context=TimeContext.OFF_PEAK,
)

print(request)

## 3 · Compute the isochrone

`compute()` auto-selects the routing backend:
- If `ORS_API_KEY` is set → OpenRouteService (cloud, faster polygon)
- Otherwise → osmnx (local Overpass download, no key needed)


In [ ]:
result = compute(request)  # uses osmnx if no ORS_API_KEY is set
print(f"Router used: {result.router_name}")

## 4 · Inspect the result GeoDataFrames

In [ ]:
# The isochrone polygon is always present
result.isochrone_gdf

In [ ]:
# Road network edges within the polygon (None if router didn't produce one)
if result.network_gdf is not None:
    print(f"{len(result.network_gdf)} reachable road segments")
    result.network_gdf.head(3)
else:
    print("network_gdf not available for this router / layer config")

In [ ]:
# Named POIs within the polygon (None if not fetched)
if result.poi_gdf is not None:
    print(f"{len(result.poi_gdf)} named places within the isochrone")
    result.poi_gdf.head(10)
else:
    print("poi_gdf not available for this router / layer config")

## 5 · Quick area check

In [ ]:
# Reproject to a metric CRS for an accurate area figure
iso_proj = result.isochrone_gdf.to_crs(result.isochrone_gdf.estimate_utm_crs())
area_km2 = iso_proj.geometry.area.iloc[0] / 1e6
print(f"Reachable area: {area_km2:.2f} km²")

## 6 · Render all four layers

In [ ]:
m = viz.render(result)
m  # display inline in Jupyter

## 7 · Render a subset of layers

You can pass an explicit list to `viz.render()` to show only what you need.


In [ ]:
m_simple = viz.render(result, layers=[VizLayer.ISOCHRONE, VizLayer.HEATMAP])
m_simple

## 8 · Save to HTML

In [ ]:
from pathlib import Path

out = Path("/mnt/team/idd/pub/idd-travel-time/maps/knh_walking_20min.html")
saved = viz.save(result, out)
print(f"Map saved to {saved}")

## 9 · Driving isochrone for comparison

Same origin, driving mode, 10 minutes — shows how much more area is reachable
by car even in a short time window.


In [ ]:
request_drive = TravelTimeRequest(
    origin_lat=-1.3006,
    origin_lon=36.8073,
    mode=TransportMode.DRIVING,
    duration_minutes=10,
    time_context=TimeContext.RUSH_HOUR,
)

result_drive = compute(request_drive)
out_drive = Path("/mnt/team/idd/pub/idd-travel-time/maps/knh_driving_10min.html")
saved_drive = viz.save(result_drive, out_drive)
print(f"Map saved to {saved_drive}")
m_drive = viz.render(result_drive, layers=[VizLayer.ISOCHRONE])
m_drive

## 10 · CLI equivalent

The same walking isochrone can be produced from the terminal:

```bash
run-isochrone \
    --lat -1.3006 --lon 36.8073 \
    --mode walking \
    --duration 20 \
    --output /mnt/team/idd/pub/idd-travel-time/maps/knh_walking_20min.html
```
